# Vakıf Instagram Hesabı: İçerik Performansı Analizi

Bir vakfın Instagram hesabında **12 Haziran – 18 Ağustos 2026** arasında paylaşılan **30 gönderinin** hangi özelliklerinin daha yüksek etkileşimle birlikte görüldüğünü inceleyen pilot çalışma.

**Araştırma soruları**
1. Gönderi formatı (Carousel, Reels, Photo) ve konusu beğeni ile ilişkili mi?
2. Etiketlenen hesap sayısı, hashtag ve harekete geçirici çağrı (CTA) kullanımı beğeniyle ilişkili mi?

**Yöntem:** Sosyal medya verileri uç değerler içerdiğinden medyan ve çeyrekler arası aralık (IQR) raporlanmış, karşılaştırmalar parametre dışı testlerle (Spearman, Mann–Whitney U, Kruskal–Wallis) yapılmıştır.

> Vakfın adı ve gönderi bağlantıları gizlilik nedeniyle veri setinden çıkarılmıştır.

## 1. Kurulum ve veri

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

os.makedirs("figures", exist_ok=True)

MAVI = "#2a78d6"      # tek seri rengi
YAZI = "#52514e"      # ikincil yazı rengi
sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e5e0", "axes.edgecolor": "#c9c8c2",
                                     "axes.labelcolor": YAZI, "xtick.color": YAZI, "ytick.color": YAZI})

df = pd.read_excel("veri.xlsx")
print(df.shape)
df.head()

Her satır bir gönderi. Değişkenlerin tanımları `README.md` dosyasındaki değişken sözlüğündedir. Beğeni sayıları **18.08.2026** tarihinde toplanmıştır; `post_age` gönderinin o tarihteki yaşını (gün) gösterir.

## 2. Genel görünüm

In [ ]:
print(f"Gönderi sayısı: {len(df)}")
print(f"Medyan beğeni: {df['likes'].median():.0f}  (IQR: {df['likes'].quantile(.25):.0f}–{df['likes'].quantile(.75):.0f})")
print(f"Ortalama beğeni: {df['likes'].mean():.0f}")
print(f"Medyan yorum: {df['comments'].median():.0f}")
en_yuksek = df['likes'].max()
print(f"En çok beğeni alan gönderinin toplamdaki payı: %{en_yuksek / df['likes'].sum() * 100:.0f}")

Ortalama (277) ile medyan (41) arasındaki büyük fark, tek bir viral gönderiden kaynaklanıyor. Bu yüzden analiz boyunca **medyan** kullanılmıştır.

## 3. Özet tablolar: gruplara göre beğeni

In [ ]:
def ozet(degisken):
    return (df.groupby(degisken)["likes"]
              .agg(n="count", medyan="median",
                   Q1=lambda x: x.quantile(.25), Q3=lambda x: x.quantile(.75))
              .assign(IQR=lambda t: t.Q3 - t.Q1)
              .sort_values("medyan", ascending=False)
              .round(2))

ozet("content_type")

In [ ]:
ozet("topic")

In [ ]:
ozet("weekday")

In [ ]:
ozet("CTA")

## 4. İstatistiksel testler

In [ ]:
# Spearman: etiketlenen hesap sayısı ile beğeni
rs, p_rs = stats.spearmanr(df["tagged_accounts"], df["likes"])
print(f"Spearman (etiket - beğeni):  rs = {rs:.3f}, p = {p_rs:.4f}")

rs_h, p_h = stats.spearmanr(df["hashtag_count"], df["likes"])
print(f"Spearman (hashtag - beğeni): rs = {rs_h:.3f}, p = {p_h:.4f}")

# Mann-Whitney U: CTA var / yok
U, p_mw = stats.mannwhitneyu(df.loc[df["CTA"] == 1, "likes"],
                             df.loc[df["CTA"] == 0, "likes"], alternative="two-sided")
print(f"Mann-Whitney (CTA):          U = {U}, p = {p_mw:.4f}")

# Kruskal-Wallis: Carousel / Reels / Photo
H, p_kw = stats.kruskal(*[g["likes"] for _, g in df.groupby("content_type")])
print(f"Kruskal-Wallis (format):     H = {H:.3f}, p = {p_kw:.4f}")

Kruskal–Wallis yalnızca "en az bir grup farklı" der. Hangi grupların farklı olduğunu bulmak için ikili Mann–Whitney testleri yapılmış, çoklu karşılaştırma nedeniyle p değerlerine **Bonferroni düzeltmesi** uygulanmıştır.

In [ ]:
turler = ["Carousel", "Reels", "Photo"]
sonuclar = []
for a, b in combinations(turler, 2):
    _, p = stats.mannwhitneyu(df.loc[df.content_type == a, "likes"],
                              df.loc[df.content_type == b, "likes"], alternative="two-sided")
    sonuclar.append({"karşılaştırma": f"{a} – {b}", "p": round(p, 3),
                     "Bonferroni p": round(min(p * 3, 1), 3)})
pd.DataFrame(sonuclar)

## 5. Grafikler

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sira = ["Carousel", "Reels", "Photo"]
sns.boxplot(data=df, x="content_type", y="likes", order=sira, color="#eceae4",
            showfliers=False, linewidth=1, ax=ax)
sns.stripplot(data=df, x="content_type", y="likes", order=sira, color=MAVI,
              size=8, jitter=0.12, edgecolor="white", linewidth=1, ax=ax)
ax.set_yscale("log")
ax.set_title(f"Gönderi türüne göre beğeni (Kruskal–Wallis p = {p_kw:.3f})", loc="left")
ax.set_xlabel("")
ax.set_ylabel("Beğeni (log ölçek)")
ax.set_xticks(range(3), [f"{t}\n(n={(df.content_type == t).sum()})" for t in sira])
fig.savefig("figures/tur_begeni.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.stripplot(data=df, x="tagged_accounts", y="likes", color=MAVI, size=8,
              jitter=0.15, edgecolor="white", linewidth=1, ax=ax)
ax.set_yscale("log")
ax.set_title(f"Etiketlenen hesap sayısı ve beğeni (Spearman rs = {rs:.2f}, p < 0,001)", loc="left")
ax.set_xlabel("Etiketlenen hesap sayısı")
ax.set_ylabel("Beğeni (log ölçek)")
fig.savefig("figures/etiket_begeni.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
t = ozet("topic").sort_values("medyan")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(t.index, t["medyan"], color=MAVI, height=0.6)
for i, (m, n) in enumerate(zip(t["medyan"], t["n"])):
    ax.text(m + 1, i, f"{m:.0f}  (n={n})", va="center", color=YAZI, fontsize=9)
ax.set_title("Konuya göre medyan beğeni", loc="left")
ax.set_xlabel("Medyan beğeni")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, t["medyan"].max() * 1.25)
fig.savefig("figures/konu_medyan.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Sonuçları dışa aktar

In [ ]:
with pd.ExcelWriter("sonuclar.xlsx") as yazici:
    for degisken in ["content_type", "topic", "weekday", "CTA", "caption_length"]:
        ozet(degisken).to_excel(yazici, sheet_name=degisken)
print("sonuclar.xlsx kaydedildi")

## 7. Bulgular ve sınırlılıklar

**Bulgular**
- Tipik bir gönderi **41 beğeni** alıyor; yorum neredeyse yok (medyan 0).
- **Etiketlenen hesap sayısı** beğeniyle orta düzeyde, anlamlı ve pozitif ilişkili (rs = 0,59; p < 0,001). Etiketli gönderilerin hiçbiri 33 beğeninin altında kalmadı.
- **Carousel** gönderileri daha yüksek beğeniyle birlikte görülüyor (Kruskal–Wallis p = 0,029); ikili karşılaştırmada anlamlı fark yalnızca Carousel ile Photo arasında (Bonferroni p = 0,048).
- **CTA** içeren ve içermeyen gönderiler arasında fark bulunamadı (p = 0,83).

**Sınırlılıklar**
- Örneklem küçük (n = 30); Reels (4) ve Photo (3) grupları çok küçük.
- Veriler gözlemsel: bulgular ilişki gösterir, nedensellik kanıtlamaz.
- Beğeniler tek bir tarihte toplandı; en yeni gönderiler etkileşim toplamaya devam ediyordu.
- Erişim, kaydetme ve takipçi sayısı gibi Instagram Insights verilerine erişim yoktu.

**Sonraki adımlar:** veri setini genişletmek, yeni gönderileri 7. ve 30. günde ölçmek, veri büyüdüğünde negatif binom regresyonu ile çok değişkenli analize geçmek.